# Smart Demand Signals — Inibsa · Interhack BCN 2026

> **Objectiu:** Generar una llista diària prioritzada d'alertes comercials `(client, família, motiu)` perquè l'equip de vendes sàpiga exactament a qui contactar, per quin producte i per quin canal.

## Arquitectura del sistema

```
master_commodities.csv ──► Motor A (Commodities)
                                ├─ Feature engineering mensual
                                ├─ KMeans segmentació comportamental
                                ├─ Cicle de reposició per client
                                └─ Alertes: reposició / fuga / captura

master_technicals.csv  ──► Motor B (Tècnics)
                                ├─ Patró individual (freq + vol)
                                ├─ Puntuació d'anomalia (z-score)
                                └─ Alertes: silenci anòmal

                    ──► Priorització unificada d'alertes
                              (impacte × urgència × probabilitat)
```

## 0. Setup & Càrrega de dades

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import ipywidgets as widgets
from IPython.display import display, clear_output
import json

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

# ── Constants globals ─────────────────────────────────────────────────────────
REFERENCE_DATE = pd.Timestamp('2025-12-30')   # "avui" = endemà de l'últim registre
CUTOFF_12M     = REFERENCE_DATE - pd.DateOffset(months=12)
CUTOFF_3M      = REFERENCE_DATE - pd.DateOffset(months=3)
CUTOFF_24M     = REFERENCE_DATE - pd.DateOffset(months=24)

# Colors de segment (reutilitzats a totes les visualitzacions)
SEG_COLORS = {
    'fidel':            '#1565C0',
    'fidel_decreixent': '#42A5F5',
    'promiscu':         '#F57C00',
    'marginal':         '#9E9E9E',
    'en_risc':          '#C62828',
    'esporàdic':        '#66BB6A',
    'perdut':           '#37474F',
    'actiu_regular':    '#1565C0',
    'actiu_esporadic':  '#F57C00',
    'inactiu_recent':   '#C62828',
    'inactiu_llarg':    '#37474F',
}

print(f'✅ Setup complet — Data de referència: {REFERENCE_DATE.date()}')

In [ ]:
comm = pd.read_csv('data/master_commodities.csv', parse_dates=['Fecha'])
tech = pd.read_csv('data/master_technicals.csv', parse_dates=['Fecha'])

print(f'Commodities: {comm.shape[0]:,} files, {comm.shape[1]} columnes')
print(f'Tècnics:     {tech.shape[0]:,} files, {tech.shape[1]} columnes')
print(f'Rang de dates: {min(comm.Fecha.min(), tech.Fecha.min()).date()} → {max(comm.Fecha.max(), tech.Fecha.max()).date()}')
print(f'Clíniques úniques (commodities): {comm.Id_Cliente.nunique():,}')
print(f'Clíniques úniques (tècnics):     {tech.Id_Cliente.nunique():,}')

## 1. EDA ràpid — Sanity checks

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Vendes mensuals commodities (excl. devolucions)
comm_valid = comm[comm.es_devolucion == 0]
monthly_comm = comm_valid.groupby(comm_valid.Fecha.dt.to_period('M'))['Valores_H'].sum() / 1000
monthly_comm.plot(ax=axes[0], color='steelblue')
axes[0].set_title('Facturació mensual — Commodities (k€)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

# Vendes mensuals tècnics
tech_valid = tech[tech.es_devolucion == 0]
monthly_tech = tech_valid.groupby(tech_valid.Fecha.dt.to_period('M'))['Valores_H'].sum() / 1000
monthly_tech.plot(ax=axes[1], color='coral')
axes[1].set_title('Facturació mensual — Tècnics (k€)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

# Distribució del potencial per família
pot_data = []
for df, label in [(comm_valid, 'Anestesia'), (comm_valid[comm_valid.Familia_Potencial == 'Bioseguridad'], 'Bioseguridad')]:
    subset = df[df.Familia_Potencial == label] if label != 'Bioseguridad' else comm_valid[comm_valid.Familia_Potencial == 'Bioseguridad']
    pot_data.append({'Familia': label, 'Potencial_mig_k': subset.groupby('Id_Cliente')['Potencial_EUR'].mean().mean() / 1000})
tech_pot = tech_valid.groupby('Id_Cliente')['Potencial_EUR'].mean().mean() / 1000
pot_data.append({'Familia': 'Biomateriales', 'Potencial_mig_k': tech_pot})
pot_df = pd.DataFrame(pot_data)
axes[2].bar(pot_df['Familia'], pot_df['Potencial_mig_k'], color=['steelblue', 'seagreen', 'coral'])
axes[2].set_title('Potencial anual mig per client (k€)')
axes[2].set_ylabel('k€')

plt.tight_layout()
plt.show()

print(f'\nDevolucions commodities: {comm[comm.es_devolucion==1].shape[0]:,} ({100*comm[comm.es_devolucion==1].shape[0]/len(comm):.1f}%)')
print(f'Línies en campanya (commodities): {comm[comm.en_campana==1].shape[0]:,} ({100*comm[comm.en_campana==1].shape[0]/len(comm):.1f}%)')

## 2. Motor A — Commodities

### 2.1 Baseline mensual (excloent devolucions i campanyes)

In [ ]:
# Baseline: excloure devolucions i períodes de campanya
comm_base = comm[(comm.es_devolucion == 0) & (comm.en_campana == 0)].copy()

# Agregació mensual per (client, família)
comm_base['year_month'] = comm_base.Fecha.dt.to_period('M')

monthly = comm_base.groupby(['Id_Cliente', 'Familia_Potencial', 'year_month']).agg(
    euros_venuts=('Valores_H', 'sum'),
    num_pedidos=('Num.Fact', 'nunique'),
    unitats=('Unidades', 'sum'),
    potencial_eur=('Potencial_EUR', 'first')
).reset_index()

monthly['year_month_dt'] = monthly['year_month'].dt.to_timestamp()

print(f'Agregació mensual: {len(monthly):,} files')
print(f'Parelles (client, família) úniques: {monthly.groupby(["Id_Cliente","Familia_Potencial"]).ngroups:,}')
monthly.head()

### 2.2 Share of Wallet — últims 12 mesos

In [ ]:
cutoff_12m = REFERENCE_DATE - pd.DateOffset(months=12)

last_12m = monthly[monthly.year_month_dt >= cutoff_12m]

sow = last_12m.groupby(['Id_Cliente', 'Familia_Potencial']).agg(
    euros_12m=('euros_venuts', 'sum'),
    potencial_eur=('potencial_eur', 'first'),
    mesos_actius_12m=('year_month', 'nunique'),
    num_pedidos_12m=('num_pedidos', 'sum')
).reset_index()

sow['share_of_wallet'] = (sow['euros_12m'] / sow['potencial_eur']).clip(upper=1.5)  # cap 150%

# Distribució del SoW
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for fam, color in [('Anestesia', 'steelblue'), ('Bioseguridad', 'seagreen')]:
    subset = sow[sow.Familia_Potencial == fam]['share_of_wallet']
    ax1.hist(subset.clip(0, 1.5), bins=40, alpha=0.6, label=fam, color=color, density=True)
ax1.set_title('Distribució Share of Wallet — Commodities')
ax1.set_xlabel('Share of Wallet (0 = tot a competència, 1 = tot a Inibsa)')
ax1.axvline(0.2, color='red', ls='--', alpha=0.5, label='20% (llindar marginal)')
ax1.axvline(0.7, color='orange', ls='--', alpha=0.5, label='70% (llindar fidel)')
ax1.legend(fontsize=8)

# Euros recuperables per família
sow['euros_recuperables'] = ((1 - sow['share_of_wallet'].clip(0, 1)) * sow['potencial_eur']).clip(lower=0)
recup = sow.groupby('Familia_Potencial')['euros_recuperables'].sum() / 1e6
recup.plot(kind='bar', ax=ax2, color=['steelblue', 'seagreen'])
ax2.set_title('Euros recuperables totals estimats (M€)')
ax2.set_ylabel('M€')
ax2.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f"\nTotal euros recuperables estimats (commodities): {sow['euros_recuperables'].sum()/1e6:.1f} M€")

### 2.3 Feature Engineering per segmentació

Calculem features comportamentals per cada `(client, família)` al llarg de tot l'historial.

In [ ]:
# ── Precomputar lookups vectoritzats (O(n), no O(n²)) ────────────────────────

# Data de l'últim pedido per (client, família)
last_pur_lookup = (
    comm_base.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha']
    .max().reset_index()
    .rename(columns={'Fecha': 'last_purchase_date'})
)

# Euros venuts en últims 3 mesos per (client, família)
euros_3m_lookup = (
    monthly[monthly.year_month_dt >= CUTOFF_3M]
    .groupby(['Id_Cliente', 'Familia_Potencial'])['euros_venuts'].sum()
    .reset_index().rename(columns={'euros_venuts': 'euros_3m'})
)

def compute_trend(series, n_months=3):
    """Pendent de regressió lineal normalitzada sobre els últims n_months."""
    if len(series) < 2:
        return 0.0
    tail = series.tail(n_months)
    x = np.arange(len(tail))
    if x.std() == 0:
        return 0.0
    slope = np.polyfit(x, tail.values, 1)[0]
    baseline = tail.mean() if tail.mean() != 0 else 1.0
    return float(slope / abs(baseline))

# ── Bucle de features (sense accés a comm_base per fila) ─────────────────────
features_list = []

for (client_id, familia), grp in monthly.groupby(['Id_Cliente', 'Familia_Potencial']):
    grp = grp.sort_values('year_month_dt')
    potencial = grp['potencial_eur'].iloc[0]

    grp_12m = grp[grp.year_month_dt >= CUTOFF_12M]
    euros_12m  = grp_12m['euros_venuts'].sum()
    sow_12m    = min(euros_12m / potencial, 1.5) if potencial > 0 else 0.0
    mesos_actius = int((grp_12m['euros_venuts'] > 0).sum())

    series_mensual = grp.set_index('year_month_dt')['euros_venuts'].resample('ME').sum()
    trend = compute_trend(series_mensual, n_months=3)

    cv = (grp_12m['euros_venuts'].std() / grp_12m['euros_venuts'].mean()
          if len(grp_12m) > 1 and grp_12m['euros_venuts'].mean() > 0 else 1.0)

    features_list.append({
        'Id_Cliente':       client_id,
        'Familia_Potencial': familia,
        'potencial_eur':    potencial,
        'euros_12m':        euros_12m,
        'share_of_wallet':  sow_12m,
        'mesos_actius_12m': mesos_actius,
        'trend_3m':         trend,
        'cv_mensual':       min(cv, 5.0),
    })

features = pd.DataFrame(features_list)

# Join vectoritzat (substitueix el filtre row-by-row que era O(n²))
features = (
    features
    .merge(last_pur_lookup, on=['Id_Cliente', 'Familia_Potencial'], how='left')
    .merge(euros_3m_lookup, on=['Id_Cliente', 'Familia_Potencial'], how='left')
)
features['euros_3m']         = features['euros_3m'].fillna(0)
features['last_purchase_date'] = pd.to_datetime(features['last_purchase_date'])
features['dies_sense_compra'] = (
    (REFERENCE_DATE - features['last_purchase_date']).dt.days.fillna(999).astype(int)
)

print(f'✅ Features: {len(features):,} parelles (client, família)')
print(f'   mesos_actius_12m → min={features.mesos_actius_12m.min()}, '
      f'mig={features.mesos_actius_12m.mean():.1f}, max={features.mesos_actius_12m.max()}')
print(f'   share_of_wallet  → min={features.share_of_wallet.min():.2f}, '
      f'mig={features.share_of_wallet.mean():.2f}, max={features.share_of_wallet.max():.2f}')
features.describe().loc[['mean', 'std', 'min', 'max']]

### 2.4 Segmentació amb K-Means (model original)

En comptes d'aplicar regles fixes, usem **K-Means** sobre les features comportamentals per identificar segments naturals. Això permet que el model s'adapti als patrons reals de les dades sense thresholds arbitraris.

In [ ]:
# Features per al clustering (normalitzades)
cluster_features = ['share_of_wallet', 'mesos_actius_12m', 'trend_3m', 'cv_mensual', 'dies_sense_compra']

# Filtrar clients amb almenys 1 compra en 24 mesos
cutoff_24m = REFERENCE_DATE - pd.DateOffset(months=24)
features_active = features[features.last_purchase_date >= cutoff_24m].copy()

X = features_active[cluster_features].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Selecció del k òptim amb el mètode del colze + silhouette
inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels, sample_size=5000, random_state=42))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(K_range, inertias, 'bo-')
ax1.set_xlabel('Nombre de clusters (k)')
ax1.set_ylabel('Inèrcia')
ax1.set_title('Mètode del colze')

ax2.plot(K_range, silhouettes, 'rs-')
ax2.set_xlabel('Nombre de clusters (k)')
ax2.set_ylabel('Silhouette score')
ax2.set_title('Silhouette score per k')

plt.tight_layout()
plt.show()

best_k = list(K_range)[np.argmax(silhouettes)]
print(f'k òptim per silhouette: {best_k} (score={max(silhouettes):.3f})')

In [ ]:
N_CLUSTERS = best_k if best_k >= 4 else 5

km_final = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=20)
features_active = features_active.copy()
features_active['cluster_raw'] = km_final.fit_predict(X_scaled)

# ── Perfil de cada cluster ────────────────────────────────────────────────────
cluster_profile = (
    features_active
    .groupby('cluster_raw')[cluster_features + ['potencial_eur']]
    .mean().round(3)
)
cluster_profile['n_clients'] = features_active.groupby('cluster_raw').size()

# ── Assignació rank-based (adaptativa a la distribució real) ─────────────────
# Problema amb thresholds fixos: si cap cluster té mesos_actius >= 4, tots
# serien "esporàdic". Solució: ordenar clusters per puntuació relativa.

def assign_segments_to_clusters(profile_df):
    """
    Assigna labels de negoci als clusters de KMeans de forma adaptativa:
    - No depèn de thresholds absoluts
    - Ordena per puntuació composta (SoW 50% + activitat 30% + recència 20%)
    - Assigna segments per percentil de rang
    """
    df = profile_df.copy()
    max_mesos = max(df['mesos_actius_12m'].max(), 1)
    max_dies  = max(df['dies_sense_compra'].max(), 1)

    df['_score'] = (
        df['share_of_wallet'] * 0.50 +
        (df['mesos_actius_12m'] / max_mesos) * 0.30 +
        (1.0 - df['dies_sense_compra'] / max_dies) * 0.20
    )
    df_sorted = df.sort_values('_score', ascending=False)
    n = len(df_sorted)
    dies_q70 = df['dies_sense_compra'].quantile(0.70)

    seg_map = {}
    for rank, (cluster_id, row) in enumerate(df_sorted.iterrows()):
        rank_pct = rank / max(n - 1, 1)   # 0.0 = millor cluster, 1.0 = pitjor
        sow   = row['share_of_wallet']
        trend = row['trend_3m']
        dies  = row['dies_sense_compra']

        if dies >= dies_q70 and rank > 0:
            seg = 'en_risc'
        elif rank_pct <= 0.25:
            seg = 'fidel' if trend >= -0.15 else 'fidel_decreixent'
        elif rank_pct <= 0.55:
            seg = 'promiscu'
        elif rank_pct <= 0.80:
            seg = 'marginal' if sow < 0.15 else 'esporàdic'
        else:
            seg = 'esporàdic'

        seg_map[cluster_id] = seg

    return seg_map

cluster_to_segment = assign_segments_to_clusters(cluster_profile)
features_active = features_active.copy()
features_active['segment'] = features_active['cluster_raw'].map(cluster_to_segment)

# Clients inactius (sense compra en 24m) → perdut
features_inactive = features[features.last_purchase_date < CUTOFF_24M].copy()
features_inactive['cluster_raw'] = -1
features_inactive['segment'] = 'perdut'
features_all = pd.concat([features_active, features_inactive], ignore_index=True)

# ── Taula de perfil de clusters (interpretable) ───────────────────────────────
cluster_profile['segment'] = cluster_profile.index.map(cluster_to_segment)
profile_display = (
    cluster_profile
    [['segment', 'share_of_wallet', 'mesos_actius_12m', 'trend_3m',
      'dies_sense_compra', 'potencial_eur', 'n_clients']]
    .rename(columns={
        'share_of_wallet':  'SoW mig',
        'mesos_actius_12m': 'Mesos actius 12m',
        'trend_3m':         'Tendència 3m',
        'dies_sense_compra':'Dies sense compra',
        'potencial_eur':    'Potencial €',
        'n_clients':        'Nº parelles'
    })
    .sort_values('SoW mig', ascending=False)
)
print(f'KMeans k={N_CLUSTERS} — Perfil de clusters:')
display(profile_display)

print('\nDistribució de segments (clients actius + inactius):')
seg_dist = features_all.groupby('segment').agg(
    n_parelles=('Id_Cliente', 'count'),
    sow_mig=('share_of_wallet', 'mean'),
    potencial_total_k=('potencial_eur', lambda x: x.sum() / 1000)
).round(2).sort_values('sow_mig', ascending=False)
display(seg_dist)

In [ ]:
# Visualització dels segments
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

seg_colors = {
    'fidel': '#2196F3',
    'fidel_decreixent': '#03A9F4',
    'promiscu': '#FF9800',
    'marginal': '#9E9E9E',
    'en_risc': '#F44336',
    'esporàdic': '#8BC34A',
    'perdut': '#37474F'
}

seg_order = ['fidel', 'fidel_decreixent', 'promiscu', 'esporàdic', 'marginal', 'en_risc', 'perdut']
counts = features_all['segment'].value_counts().reindex([s for s in seg_order if s in features_all['segment'].unique()], fill_value=0)
colors = [seg_colors.get(s, 'gray') for s in counts.index]

ax1.barh(counts.index, counts.values, color=colors)
ax1.set_title('Nombre de parelles (client, família) per segment')
ax1.set_xlabel('Nombre')

# Share of wallet per segment (boxplot)
plot_data = features_all[features_all.segment.isin(seg_order)]
seg_present = [s for s in seg_order if s in plot_data['segment'].unique()]
data_by_seg = [plot_data[plot_data.segment == s]['share_of_wallet'].values for s in seg_present]
bp = ax2.boxplot(data_by_seg, labels=seg_present, patch_artist=True)
for patch, seg in zip(bp['boxes'], seg_present):
    patch.set_facecolor(seg_colors.get(seg, 'gray'))
    patch.set_alpha(0.7)
ax2.set_title('Share of Wallet per segment')
ax2.set_ylabel('Share of Wallet')
ax2.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### 2.5 Cicle de reposició per client

Per a clients `fidel` i `promiscu`: calculem el cicle mitjà entre pedidos per detectar quan toca la propera compra.

In [ ]:
# ── Versió vectoritzada: un sol groupby sobre tot comm_base ──────────────────
# Pas 1: data mínima per factura × (client, família) → un pedido = una fila
pedidos_all = (
    comm_base
    .groupby(['Id_Cliente', 'Familia_Potencial', 'Num.Fact'], sort=False)['Fecha']
    .min()
    .reset_index()
    .sort_values(['Id_Cliente', 'Familia_Potencial', 'Fecha'])
)

# Pas 2: gap entre pedidos consecutius dins de cada (client, família)
pedidos_all['gap_dies'] = (
    pedidos_all
    .groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha']
    .diff()
    .dt.days
)

# Pas 3: filtrar gaps vàlids (>0 per descartar factures del mateix dia)
valid_gaps = pedidos_all[pedidos_all['gap_dies'] > 0]

# Pas 4: estadístiques per (client, família)
cycle_stats = (
    valid_gaps
    .groupby(['Id_Cliente', 'Familia_Potencial'])['gap_dies']
    .agg(cicle_mig_dies='mean', cicle_std_dies='std')
    .reset_index()
)

# ── Unir amb els candidats de segmentació ────────────────────────────────────
target_segments = ['fidel', 'fidel_decreixent', 'promiscu']
cycle_candidates = features_all[features_all.segment.isin(target_segments)].copy()

cycle_candidates = cycle_candidates.merge(
    cycle_stats,
    on=['Id_Cliente', 'Familia_Potencial'],
    how='left'
)

# Assegurem que last_purchase_date és Timestamp (pot ser date o NaT)
cycle_candidates['last_purchase_date'] = pd.to_datetime(cycle_candidates['last_purchase_date'])

# Pas 5: data esperada del proper pedido i dies de retard
mask_valid = cycle_candidates['cicle_mig_dies'].notna()
cycle_candidates['proper_pedido_esperat'] = pd.NaT
cycle_candidates.loc[mask_valid, 'proper_pedido_esperat'] = (
    cycle_candidates.loc[mask_valid, 'last_purchase_date'] +
    pd.to_timedelta(cycle_candidates.loc[mask_valid, 'cicle_mig_dies'], unit='D')
)

cycle_candidates['dies_retard'] = (
    (REFERENCE_DATE - cycle_candidates['proper_pedido_esperat'])
    .dt.days
    .clip(lower=0)
)

valid_cycles = cycle_candidates.dropna(subset=['cicle_mig_dies'])
print(f'Cicles de reposició calculats per {len(valid_cycles):,} parelles (client, família)')
print(f'Cicle mig general: {valid_cycles.cicle_mig_dies.mean():.0f} dies')
print(f'Clients amb retard > 0 dies: {(valid_cycles.dies_retard > 0).sum():,}')

# Distribució del cicle de reposició
fig, ax = plt.subplots(figsize=(10, 4))
for fam, color in [('Anestesia', 'steelblue'), ('Bioseguridad', 'seagreen')]:
    data = valid_cycles[valid_cycles.Familia_Potencial == fam]['cicle_mig_dies']
    ax.hist(data.clip(0, 180), bins=50, alpha=0.6, label=f'{fam} (n={len(data):,})', color=color, density=True)
ax.set_title('Distribució del cicle de reposició mig (dies)')
ax.set_xlabel('Dies entre pedidos')
ax.legend()
plt.tight_layout()
plt.show()

### 2.6 Generació d'alertes — Motor Commodities

In [ ]:
def generate_commodity_alerts(features_all, cycle_candidates, reference_date):
    alerts = []
    
    # ── Alertes basades en cicle (fidels + promiscus amb cicle calculat) ──
    for _, row in cycle_candidates.dropna(subset=['cicle_mig_dies']).iterrows():
        cicle = row['cicle_mig_dies']
        std = row['cicle_std_dies'] if not pd.isna(row['cicle_std_dies']) else cicle * 0.3
        retard = row['dies_retard']
        sow = row['share_of_wallet']
        seg = row['segment']
        
        # Finestra d'oportunitat de captura per a promiscus
        if seg == 'promiscu' and abs(retard) <= 3 and retard >= -3:
            # El pedido és imminent
            impacte = row['potencial_eur'] * (1 - sow)
            urgencia = 'alta'
            tipo = 'finestra_captura'
            motiu = (
                f"Client promiscu de {row['Familia_Potencial']}. "
                f"Cicle habitual: {cicle:.0f} dies. "
                f"La seva finestra de compra és ARA (retard: {retard:.0f} dies). "
                f"Share of wallet actual: {sow*100:.0f}%. "
                f"Oportunitat de capturar {impacte:.0f}€ de quota de mercat."
            )
        elif retard > cicle * 0.5 and seg in ['fidel', 'fidel_decreixent']:
            # Client fidel que porta massa temps sense comprar
            impacte = row['euros_12m'] / 12  # valor mensual estimat
            if retard > cicle + 1.5 * std:
                urgencia = 'critica'
                tipo = 'risc_fuga'
            elif retard > cicle + 0.5 * std:
                urgencia = 'alta'
                tipo = 'risc_fuga'
            else:
                urgencia = 'mitjana'
                tipo = 'reposicio_pendent'
            motiu = (
                f"Client {seg} de {row['Familia_Potencial']}. "
                f"Cicle habitual: {cicle:.0f} dies (±{std:.0f}). "
                f"Porta {row['dies_sense_compra']:.0f} dies sense comprar ({retard:.0f} dies de retard). "
                f"Share of wallet últims 12m: {sow*100:.0f}%. "
                f"Possible {'inici de fuga' if tipo == 'risc_fuga' else 'retard de reposició'}."
            )
        elif retard > 0 and seg == 'promiscu':
            impacte = row['potencial_eur'] * (1 - sow) * 0.3  # oportunitat parcial
            urgencia = 'baixa'
            tipo = 'reposicio_pendent'
            motiu = (
                f"Client promiscu de {row['Familia_Potencial']}. "
                f"El cicle de reposició indica compra pendent ({retard:.0f} dies de retard). "
                f"Oportunitat de capturar part de la compra."
            )
        else:
            continue  # Sense alerta
        
        alerts.append({
            'id_client': int(row['Id_Cliente']),
            'familia': row['Familia_Potencial'],
            'bloc_analitic': 'Commodities',
            'tipus_alerta': tipo,
            'segment_client': seg,
            'dies_sense_compra': int(row['dies_sense_compra']),
            'cicle_habitual_dies': round(cicle, 0),
            'dies_retard': int(retard),
            'share_of_wallet_12m': round(sow, 3),
            'potencial_anual_eur': round(row['potencial_eur'], 2),
            'impacte_recuperable_eur': round(impacte, 2),
            'urgencia': urgencia,
            'motiu_explicat': motiu,
            'data_alerta': reference_date.strftime('%Y-%m-%d')
        })
    
    # ── Alertes per clients en risc (segment en_risc) sense cicle calculat ──
    en_risc = features_all[
        (features_all.segment == 'en_risc') &
        (~features_all.Id_Cliente.isin(cycle_candidates.Id_Cliente))
    ]
    for _, row in en_risc.iterrows():
        sow = row['share_of_wallet']
        impacte = row['euros_12m'] * 0.5  # estimat si es recupera
        alerts.append({
            'id_client': int(row['Id_Cliente']),
            'familia': row['Familia_Potencial'],
            'bloc_analitic': 'Commodities',
            'tipus_alerta': 'en_risc',
            'segment_client': 'en_risc',
            'dies_sense_compra': int(row['dies_sense_compra']),
            'cicle_habitual_dies': None,
            'dies_retard': None,
            'share_of_wallet_12m': round(sow, 3),
            'potencial_anual_eur': round(row['potencial_eur'], 2),
            'impacte_recuperable_eur': round(impacte, 2),
            'urgencia': 'alta',
            'motiu_explicat': (
                f"Client en risc de {row['Familia_Potencial']}. "
                f"Porta {row['dies_sense_compra']:.0f} dies sense comprar."
            ),
            'data_alerta': reference_date.strftime('%Y-%m-%d')
        })
    
    return pd.DataFrame(alerts)

alerts_comm = generate_commodity_alerts(features_all, cycle_candidates, REFERENCE_DATE)
print(f'Alertes generades (Commodities): {len(alerts_comm):,}')
print(alerts_comm['tipus_alerta'].value_counts())

## 3. Motor B — Productes Tècnics (Biomaterials)

### 3.1 Patró individual per client

In [ ]:
# ── Motor B completament vectoritzat (sense loop per client) ──────────────────
tech_base = tech[(tech.es_devolucion == 0) & (tech.en_campana == 0)].copy()

# Pas 1: un pedido = una fila (mínim data per factura)
tech_pedidos = (
    tech_base
    .groupby(['Id_Cliente', 'Num.Fact'], sort=False)
    .agg(data=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    .sort_values(['Id_Cliente', 'data'])
)

# Pas 2: gaps entre pedidos consecutius per client
tech_pedidos['gap_dies'] = (
    tech_pedidos.groupby('Id_Cliente')['data'].diff().dt.days
)

# Pas 3: freqüència (mig i std) per client
tech_freq = (
    tech_pedidos[tech_pedidos['gap_dies'] > 0]
    .groupby('Id_Cliente')['gap_dies']
    .agg(freq_mig_dies='mean', std_freq_dies='std')
    .reset_index()
)

# Pas 4: estadístiques generals per client
tech_agg = (
    tech_pedidos.groupby('Id_Cliente')
    .agg(
        n_pedidos_total=('Num.Fact', 'count'),
        last_purchase_date=('data', 'max'),
        vol_mig_eur=('euros', 'mean')
    ).reset_index()
)

# Pas 5: pedidos en últims 12 mesos
tech_12m = (
    tech_pedidos[tech_pedidos['data'] >= CUTOFF_12M]
    .groupby('Id_Cliente').size()
    .reset_index(name='n_pedidos_12m')
)

# Pas 6: potencial per client (nom de columna unificat → potencial_eur)
tech_pot = (
    tech_base.groupby('Id_Cliente')['Potencial_EUR']
    .first().reset_index()
    .rename(columns={'Potencial_EUR': 'potencial_eur'})
)

# Pas 7: join de tot
tech_patterns = (
    tech_agg
    .merge(tech_freq, on='Id_Cliente', how='left')
    .merge(tech_12m,  on='Id_Cliente', how='left')
    .merge(tech_pot,  on='Id_Cliente', how='left')
)
tech_patterns['n_pedidos_12m']    = tech_patterns['n_pedidos_12m'].fillna(0).astype(int)
tech_patterns['last_purchase_date'] = pd.to_datetime(tech_patterns['last_purchase_date'])
tech_patterns['dies_sense_compra'] = (REFERENCE_DATE - tech_patterns['last_purchase_date']).dt.days
tech_patterns['Familia_Potencial'] = 'Biomateriales'

# Pas 8: classificació del client tècnic
def classify_tech_client(row):
    n12, dies = row['n_pedidos_12m'], row['dies_sense_compra']
    if n12 >= 4:                         return 'actiu_regular'
    if 1 <= n12 < 4:                     return 'actiu_esporadic'
    if n12 == 0 and dies <= 365:         return 'inactiu_recent'
    return 'inactiu_llarg'

tech_patterns['tipus_client'] = tech_patterns.apply(classify_tech_client, axis=1)

# Pas 9: z-score de silenci (vectoritzat)
freq = tech_patterns['freq_mig_dies']
std  = tech_patterns['std_freq_dies'].fillna(freq * 0.30)
dies = tech_patterns['dies_sense_compra']

tech_patterns['z_score_silenci'] = np.where(
    freq.notna() & (std > 0),
    (dies - freq) / std,
    np.where(
        freq.notna() & (dies > freq),
        (dies - freq) / (freq * 0.30),
        0.0
    )
)

print(f'✅ Motor Tècnics: {len(tech_patterns):,} clients analitzats')
print(f'\nDistribució per tipus de client:')
print(tech_patterns['tipus_client'].value_counts().to_string())
print(f'\nFreqüència mig (clients amb ≥2 pedidos): '
      f'{tech_patterns.freq_mig_dies.dropna().mean():.0f} dies')

### 3.2 Detecció d'anomalies per z-score del silenci

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribució del z-score per tipus de client
for tipus, color in [('actiu_regular', '#2196F3'), ('actiu_esporadic', '#FF9800'), ('inactiu_recent', '#F44336')]:
    subset = tech_patterns[tech_patterns.tipus_client == tipus]['z_score_silenci'].clip(-3, 10)
    if len(subset) > 0:
        axes[0].hist(subset, bins=40, alpha=0.6, label=f'{tipus} (n={len(subset)})', color=color, density=True)

axes[0].axvline(1, color='orange', ls='--', label='z=1 (alerta groga)')
axes[0].axvline(2, color='red', ls='--', label='z=2 (alerta vermella)')
axes[0].set_title('Distribució del Z-score de silenci per tipus de client')
axes[0].set_xlabel('Z-score (dies sense compra vs patró habitual)')
axes[0].legend(fontsize=8)

# Scatter: freq_mig vs dies_sense_compra (colorejat per z-score)
valid = tech_patterns.dropna(subset=['freq_mig_dies'])
scatter = axes[1].scatter(
    valid['freq_mig_dies'].clip(0, 200),
    valid['dies_sense_compra'].clip(0, 400),
    c=valid['z_score_silenci'].clip(-2, 5),
    cmap='RdYlGn_r',
    alpha=0.4,
    s=10
)
axes[1].plot([0, 200], [0, 200], 'k--', alpha=0.3, label='Dies sense compra = cicle mig')
axes[1].plot([0, 200], [0, 400], 'r--', alpha=0.3, label='Dies sense compra = 2× cicle mig')
plt.colorbar(scatter, ax=axes[1], label='Z-score silenci')
axes[1].set_title('Cicle mig vs Dies sense compra')
axes[1].set_xlabel('Freqüència mig habitual (dies)')
axes[1].set_ylabel('Dies sense compra')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Llindar permissiu per a esporàdics (z>3), estàndard per a regulars (z>2)
def get_alert_level_tech(row):
    z = row['z_score_silenci']
    tipus = row['tipus_client']
    llindar_alt = 3.0 if tipus == 'actiu_esporadic' else 2.0
    llindar_groc = 2.0 if tipus == 'actiu_esporadic' else 1.0
    
    if z >= llindar_alt:
        return 'vermella'
    elif z >= llindar_groc:
        return 'groga'
    else:
        return 'cap'

tech_patterns['nivell_alerta'] = tech_patterns.apply(get_alert_level_tech, axis=1)
print('Alertes tècniques per nivell:')
print(tech_patterns['nivell_alerta'].value_counts())

### 3.3 Generació d'alertes — Motor Tècnics

In [ ]:
def generate_technical_alerts(tech_patterns, reference_date):
    alerts = []

    alertable = tech_patterns[
        (tech_patterns.nivell_alerta != 'cap') &
        (tech_patterns.tipus_client.isin(['actiu_regular', 'actiu_esporadic', 'inactiu_recent']))
    ]

    for _, row in alertable.iterrows():
        z    = row['z_score_silenci']
        freq = row['freq_mig_dies']
        std  = row['std_freq_dies']
        vol  = row['vol_mig_eur']
        pot  = row['potencial_eur']      # ← nom unificat (era 'Potencial_EUR')

        if row['nivell_alerta'] == 'vermella':
            urgencia = 'critica' if z >= 3.5 else 'alta'
            tipo     = 'silenci_anomal_crític'
        else:
            urgencia = 'mitjana'
            tipo     = 'silenci_anomal_vigilar'

        freq_str = f'{freq:.0f}' if pd.notna(freq) else 'desconegut'
        std_str  = f' ±{std:.0f}' if pd.notna(std) else ''

        impacte = vol if pd.notna(vol) else (pot * 0.05 if pd.notna(pot) else 0.0)

        motiu = (
            f"Client {row['tipus_client']} de Biomaterials. "
            f"Historial: {row['n_pedidos_total']} pedidos ({row['n_pedidos_12m']} últims 12m). "
            f"Freqüència habitual: {freq_str}{std_str} dies. "
            f"Porta {row['dies_sense_compra']:.0f} dies sense comprar (z={z:.1f}). "
            f"Anomalia {'crítica' if urgencia in ['critica', 'alta'] else 'moderada'} detectada."
        )

        alerts.append({
            'id_client':             int(row['Id_Cliente']),
            'familia':               'Biomateriales',
            'bloc_analitic':         'Productes Tècnics',
            'tipus_alerta':          tipo,
            'segment_client':        row['tipus_client'],
            'dies_sense_compra':     int(row['dies_sense_compra']),
            'cicle_habitual_dies':   round(freq, 0) if pd.notna(freq) else None,
            'z_score_silenci':       round(z, 2),
            'nivell_alerta':         row['nivell_alerta'],
            'potencial_anual_eur':   round(pot, 2)  if pd.notna(pot)  else None,
            'impacte_recuperable_eur': round(impacte, 2),
            'urgencia':              urgencia,
            'motiu_explicat':        motiu,
            'data_alerta':           reference_date.strftime('%Y-%m-%d'),
        })

    return pd.DataFrame(alerts) if alerts else pd.DataFrame()

alerts_tech = generate_technical_alerts(tech_patterns, REFERENCE_DATE)
print(f'✅ Alertes tècnics: {len(alerts_tech):,}')
if len(alerts_tech) > 0:
    print(alerts_tech['tipus_alerta'].value_counts().to_string())

## 4. Priorització Unificada d'Alertes

### 4.1 Puntuació de prioritat

```
prioritat = impacte_econòmic × urgència_temporal × probabilitat_conversió
```

In [ ]:
# Mapa d'urgència a valor numèric
urgencia_score = {
    'critica': 1.0,
    'alta': 0.75,
    'mitjana': 0.5,
    'baixa': 0.25
}

# Probabilitat de conversió estimada per segment/tipus
conversio_prob = {
    'fidel': 0.7,
    'fidel_decreixent': 0.55,
    'promiscu': 0.45,
    'en_risc': 0.30,
    'esporàdic': 0.25,
    'marginal': 0.20,
    'actiu_regular': 0.60,
    'actiu_esporadic': 0.40,
    'inactiu_recent': 0.30,
    'perdut': 0.10
}

# Canal recomanat per segment
canal_map = {
    'fidel': 'delegat',
    'fidel_decreixent': 'delegat',
    'promiscu': 'televenda',
    'en_risc': 'delegat',
    'marginal': 'marketing_automatitzat',
    'esporàdic': 'televenda',
    'actiu_regular': 'delegat',
    'actiu_esporadic': 'televenda',
    'inactiu_recent': 'delegat',
    'perdut': 'marketing_automatitzat'
}

def score_alert(row):
    impacte = row.get('impacte_recuperable_eur', 0)
    urgencia = urgencia_score.get(row.get('urgencia', 'baixa'), 0.25)
    prob_conv = conversio_prob.get(row.get('segment_client', 'esporàdic'), 0.25)
    
    # Normalitzem l'impacte (log scale per evitar que outliers dominin)
    impacte_norm = np.log1p(max(impacte, 0)) / np.log1p(50000)  # 50k€ com a referència
    
    return impacte_norm * urgencia * prob_conv

# Unificar i puntuar
all_alerts = pd.concat([alerts_comm, alerts_tech], ignore_index=True)
all_alerts['prioritat_score'] = all_alerts.apply(score_alert, axis=1)
all_alerts['canal_recomanat'] = all_alerts['segment_client'].map(canal_map).fillna('televenda')

# Ordenar per prioritat
all_alerts = all_alerts.sort_values('prioritat_score', ascending=False).reset_index(drop=True)
all_alerts['rank_prioritat'] = all_alerts.index + 1

print(f'Total alertes generades: {len(all_alerts):,}')
print(f'\nDistribució per bloc analític:')
print(all_alerts.groupby('bloc_analitic').size())
print(f'\nDistribució per urgència:')
print(all_alerts['urgencia'].value_counts())
print(f'\nImpacte econòmic total estimat: {all_alerts["impacte_recuperable_eur"].sum()/1e6:.1f} M€')

### 4.2 Top alertes del dia

In [ ]:
top_cols = ['rank_prioritat', 'id_client', 'familia', 'tipus_alerta', 'segment_client',
            'urgencia', 'dies_sense_compra', 'share_of_wallet_12m',
            'impacte_recuperable_eur', 'canal_recomanat', 'prioritat_score']

top20 = all_alerts[[c for c in top_cols if c in all_alerts.columns]].head(20)
print(f'TOP 20 alertes per a {REFERENCE_DATE.date()}:')
display(top20)

In [ ]:
# Mostra l'alerta top en format JSON (per integrar amb el CRM)
top_alert = all_alerts.iloc[0].to_dict()
# Netejar NaN per a JSON
top_alert_clean = {k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in top_alert.items()}
print('Exemple d\'alerta en format JSON (top prioritat):')
print(json.dumps(top_alert_clean, indent=2, ensure_ascii=False, default=str))

### 4.3 Visualització final — Dashboard d'alertes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(f"Smart Demand Signals — Dashboard d'alertes {REFERENCE_DATE.date()}", fontsize=14, fontweight='bold')

# 1. Alertes per canal i urgència
canal_urgencia = all_alerts.groupby(['canal_recomanat', 'urgencia']).size().unstack(fill_value=0)
canal_urgencia = canal_urgencia.reindex(columns=['critica', 'alta', 'mitjana', 'baixa'], fill_value=0)
canal_urgencia.plot(kind='bar', ax=axes[0, 0], color=['#D32F2F', '#F57C00', '#FBC02D', '#388E3C'])
axes[0, 0].set_title('Alertes per canal i urgència')
axes[0, 0].set_xlabel('')
axes[0, 0].tick_params(axis='x', rotation=20)
axes[0, 0].legend(title='Urgència', fontsize=8)

# 2. Impacte acumulat per rang
top100 = all_alerts.head(100)
top100['impacte_acumulat'] = top100['impacte_recuperable_eur'].cumsum() / 1000
axes[0, 1].plot(range(1, 101), top100['impacte_acumulat'], color='steelblue', linewidth=2)
axes[0, 1].fill_between(range(1, 101), top100['impacte_acumulat'], alpha=0.2, color='steelblue')
axes[0, 1].set_title('Impacte econòmic acumulat (Top 100 alertes)')
axes[0, 1].set_xlabel('Rang d\'alerta')
axes[0, 1].set_ylabel('Impacte acumulat (k€)')

# 3. Distribució per tipus d'alerta
tipo_counts = all_alerts['tipus_alerta'].value_counts().head(8)
tipo_colors = {'risc_fuga': '#F44336', 'finestra_captura': '#4CAF50', 'reposicio_pendent': '#FF9800',
               'silenci_anomal_crític': '#9C27B0', 'silenci_anomal_vigilar': '#E91E63', 'en_risc': '#FF5722'}
bar_colors = [tipo_colors.get(t, '#607D8B') for t in tipo_counts.index]
axes[1, 0].barh(tipo_counts.index, tipo_counts.values, color=bar_colors)
axes[1, 0].set_title('Nombre d\'alertes per tipus')
axes[1, 0].set_xlabel('Nombre')

# 4. Impacte potencial per família i urgència
impacte_fam = all_alerts.groupby(['familia', 'urgencia'])['impacte_recuperable_eur'].sum().unstack(fill_value=0) / 1000
impacte_fam = impacte_fam.reindex(columns=['critica', 'alta', 'mitjana', 'baixa'], fill_value=0)
impacte_fam.plot(kind='bar', ax=axes[1, 1], color=['#D32F2F', '#F57C00', '#FBC02D', '#388E3C'])
axes[1, 1].set_title('Impacte econòmic potencial per família (k€)')
axes[1, 1].set_xlabel('')
axes[1, 1].set_ylabel('k€')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].legend(title='Urgència', fontsize=8)

plt.tight_layout()
plt.show()

## 5. Exportació i resum final

In [ ]:
# Exportar alertes del dia
output_path = f"data/alertes_{REFERENCE_DATE.strftime('%Y%m%d')}.csv"
all_alerts.to_csv(output_path, index=False)
print(f'Alertes exportades a: {output_path}')

# Resum executiu
print('\n' + '='*60)
print(f"RESUM EXECUTIU — {REFERENCE_DATE.date()}")
print('='*60)
print(f"Total alertes generades: {len(all_alerts):,}")
print(f"  Crítiques:   {(all_alerts.urgencia=='critica').sum():>5,}")
print(f"  Altes:       {(all_alerts.urgencia=='alta').sum():>5,}")
print(f"  Mitjanes:    {(all_alerts.urgencia=='mitjana').sum():>5,}")
print(f"  Baixes:      {(all_alerts.urgencia=='baixa').sum():>5,}")
print(f"")
print(f"Per canal:")
canal_counts = all_alerts['canal_recomanat'].value_counts()
for canal, n in canal_counts.items():
    print(f"  {canal:<30} {n:>5,}")
print(f"")
print(f"Impacte econòmic total estimat:")
print(f"  Top 10 alertes:  {all_alerts.head(10)['impacte_recuperable_eur'].sum()/1000:>8.1f} k€")
print(f"  Top 50 alertes:  {all_alerts.head(50)['impacte_recuperable_eur'].sum()/1000:>8.1f} k€")
print(f"  Total:           {all_alerts['impacte_recuperable_eur'].sum()/1000:>8.1f} k€")
print('='*60)

## 6. Anàlisi de la qualitat del model

Com que no tenim etiquetes de conversió (si la intervenció va funcionar o no), validem la coherència del model amb heurístiques basades en les dades.

In [ ]:
# Validació: els clients amb alerta de risc_fuga haurien de tenir dies_sense_compra > cicle_habitual
risc_fuga = all_alerts[all_alerts.tipus_alerta == 'risc_fuga'].copy()
if len(risc_fuga) > 0 and 'cicle_habitual_dies' in risc_fuga.columns:
    risc_valid = risc_fuga.dropna(subset=['cicle_habitual_dies'])
    pct_valid = (risc_valid['dies_sense_compra'] > risc_valid['cicle_habitual_dies']).mean()
    print(f'Validació coherència risc_fuga: {pct_valid*100:.1f}% dels clients alerten han superat el seu cicle habitual')

# Concentració del valor: quantes alertes cobreixen el 80% de l'impacte?
all_alerts_sorted = all_alerts.sort_values('prioritat_score', ascending=False)
impacte_total = all_alerts_sorted['impacte_recuperable_eur'].sum()
all_alerts_sorted['impacte_cum_pct'] = all_alerts_sorted['impacte_recuperable_eur'].cumsum() / impacte_total
n_80pct = (all_alerts_sorted['impacte_cum_pct'] <= 0.8).sum()
pct_clients = n_80pct / len(all_alerts_sorted) * 100
print(f'Concentració: les primeres {n_80pct} alertes ({pct_clients:.1f}%) cobreixen el 80% de l\'impacte potencial')

# Segmentació KMeans: comprovem que els clusters reflecteixen les etiquetes
print(f'\nValidació clustering KMeans (k={N_CLUSTERS}):')
print(f'  Silhouette score final: {silhouette_score(X_scaled, km_final.labels_, sample_size=5000, random_state=42):.3f}')
print(f'  Segments identificats: {", ".join(features_active.groupby("cluster_raw")["segment"].first().values)}')

# Estadística de la consistència del z-score tècnic
z_stats = tech_patterns.dropna(subset=['z_score_silenci', 'freq_mig_dies'])
print(f'\nAnomalies tècniques detectades:')
print(f'  z > 1 (groga): {(z_stats.z_score_silenci > 1).sum():,} clients')
print(f'  z > 2 (vermella regular): {(z_stats[z_stats.tipus_client=="actiu_regular"].z_score_silenci > 2).sum():,} clients')
print(f'  z > 3 (vermella esporàdic): {(z_stats[z_stats.tipus_client=="actiu_esporadic"].z_score_silenci > 3).sum():,} clients')

---

## Conclusions i pròxims passos

### Què hem construït

| Motor | Mètode | Alertes generades |
|---|---|---|
| Motor A – Commodities | KMeans behavioural clustering + cicle de reposició | Reposició / fuga / captura |
| Motor B – Tècnics | Z-score de silenci per patró individual | Anomalia de silenci (groga/vermella) |

### Originalitat del model

- **Segmentació no supervisada (KMeans):** en lloc de thresholds manuals, el model aprèn els segments comportamentals directament de les dades.
- **Z-score de silenci personalitzat:** cada client tècnic té el seu propi patró de referència; l'anomalia és relativa a ell, no a la mitjana global.
- **Puntuació composta de prioritat:** combina impacte econòmic (log-escala), urgència temporal i probabilitat de conversió per segment.

### Millores suggerides (pròximes iteracions)

1. **Feedback loop:** registrar `alerta → acció → resultat` per calibrar `conversio_prob` amb dades reals
2. **Anàlisi geogràfica:** prioritzar per ruta de delegat (agrupació per codi postal)
3. **Model de supervivència (Kaplan-Meier):** per estimar la probabilitat de fuga en funció del temps des de l'últim pedido
4. **Gradient boosting per conversió:** si es disposa d'etiquetes (conversions passades), entrenar un model per predir la probabilitat de resposta positiva

---

## 7. Explorador Interactiu per Client

Visualitza la línia de temps de compres de qualsevol client, normalitzada a **dia 0 = primera compra**.

**Controls:**
- **Família** — Anestesia, Bioseguridad o Biomateriales
- **Client ID** — dropdown filtrat per família (clients amb ≥ 1 pedido)
- **Slider «Mínim pedidos establert»** — si el client té *menys* pedidos que el llindar → **NOU** (barres taronges, cicle poblacional); altrament → **ESTABLERT** (barres blaves, cicle personal)

**Llegenda del gràfic:**
- 🔵 Barres blaves = client establert (cicle personal calculat)
- 🟠 Barres taronges = client nou (cicle poblacional de la família)
- 🟢 Zona verda = finestra de compra esperada (cicle ± 0.5σ)
- 🔴 Zona vermella = zona de risc (retard > 0.5σ)
- ⬛ Línia negra vertical = avui

In [68]:
# ── 7.1 Construir timeline unificada (commodities + tècnics) ─────────────────

# Commodities: un pedido = una fila
comm_tl = (
    comm_base
    .groupby(['Id_Cliente', 'Familia_Potencial', 'Num.Fact'])
    .agg(Fecha=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    [['Id_Cliente', 'Familia_Potencial', 'Fecha', 'euros']]
)

# Tècnics: un pedido = una fila
tech_tl = (
    tech_base
    .groupby(['Id_Cliente', 'Num.Fact'])
    .agg(Fecha=('Fecha', 'min'), euros=('Valores_H', 'sum'))
    .reset_index()
    .assign(Familia_Potencial='Biomateriales')
    [['Id_Cliente', 'Familia_Potencial', 'Fecha', 'euros']]
)

# Unió i normalització
timeline = (
    pd.concat([comm_tl, tech_tl], ignore_index=True)
    .sort_values(['Id_Cliente', 'Familia_Potencial', 'Fecha'])
    .reset_index(drop=True)
)
timeline['primer_pedido'] = (
    timeline.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha'].transform('min')
)
timeline['dies_des_del_primer'] = (
    (timeline['Fecha'] - timeline['primer_pedido']).dt.days
)
timeline['n_pedidos_total'] = (
    timeline.groupby(['Id_Cliente', 'Familia_Potencial'])['Fecha'].transform('count')
)

# ── Cicles poblacionals (per a clients nous) ─────────────────────────────────
# valid_gaps ve de Motor A (cell 17); tech_pedidos ve de Motor B (cell 21)
pop_comm = (
    valid_gaps
    .groupby('Familia_Potencial')['gap_dies']
    .agg(pop_cicle_mig='mean', pop_cicle_std='std')
    .reset_index()
)
bio_g = tech_pedidos[tech_pedidos['gap_dies'] > 0]['gap_dies']
pop_bio = pd.DataFrame([{
    'Familia_Potencial': 'Biomateriales',
    'pop_cicle_mig': bio_g.mean() if len(bio_g) > 0 else 60.0,
    'pop_cicle_std': bio_g.std()  if len(bio_g) > 1 else 20.0,
}])
pop_cycle = pd.concat([pop_comm, pop_bio], ignore_index=True)

# tech_freq: freqüències individuals de tècnics (ve de cell 21)
# Ja existeix com a variable 'tech_freq'

print(f'✅ Timeline: {len(timeline):,} pedidos  |  '
      f'{timeline.Id_Cliente.nunique():,} clients  |  '
      f'{timeline.Familia_Potencial.nunique()} famílies')
print('\nCicles poblacionals:')
display(pop_cycle.round(1))

✅ Timeline: 102,690 pedidos  |  8,053 clients  |  3 famílies

Cicles poblacionals:


,Familia_Potencial,pop_cicle_mig,pop_cicle_std
0,Anestesia,148.70,146.20
1,Bioseguridad,139.30,165.30
2,Biomateriales,97.00,138.30


In [ ]:
# ── 7.2 Gràfic interactiu amb slider "avui simulat" (backtesting) ─────────────
#
# L'slider "Avui simulat" mou la línia negra en el temps:
#   - Barres BLAVES/TARONJA = historial conegut (fins l'avui simulat)
#   - Barres GRISES         = compres reals futures (el "ground truth")
#   - Barres VERDES         = compres reals que caient dins la zona de predicció ✅
#   - El cicle es recalcula NOMÉS amb l'historial conegut (sense data leakage)

FAMILIES = ['Anestesia', 'Bioseguridad', 'Biomateriales']

def get_clients_for_familia(familia, min_n=1):
    mask = (timeline['Familia_Potencial'] == familia) & (timeline['n_pedidos_total'] >= min_n)
    return sorted(timeline.loc[mask, 'Id_Cliente'].unique().tolist())

# ── Widgets ───────────────────────────────────────────────────────────────────
familia_w = widgets.Dropdown(
    options=FAMILIES, value='Anestesia',
    description='📦 Família:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='240px')
)
client_w = widgets.Dropdown(
    options=get_clients_for_familia('Anestesia'),
    description='🏥 Client ID:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='260px')
)
min_pedidos_w = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='📊 Mínim pedidos (establert):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='420px')
)
dia_avui_w = widgets.IntSlider(
    value=1000, min=1, max=2000, step=5,
    description='📅 Avui simulat (dies des del 1r pedido):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)
out = widgets.Output()


def _get_client_timeline(client_id, familia):
    mask = (timeline['Id_Cliente'] == client_id) & (timeline['Familia_Potencial'] == familia)
    return timeline.loc[mask].sort_values('dies_des_del_primer').copy()


def _get_cycle(known_df, familia, es_nou):
    """
    Calcula el cicle ÚNICAMENT a partir de l'historial conegut (no data leakage).
    Retorna (cicle, cicle_std, cicle_tipus, bar_color, cicle_color).
    """
    gaps = known_df['dies_des_del_primer'].diff().dropna()
    gaps = gaps[gaps > 0]

    if not es_nou and len(gaps) >= 1:
        cicle     = float(gaps.mean())
        cicle_std = float(gaps.std()) if len(gaps) > 1 else cicle * 0.30
        if pd.isna(cicle_std) or cicle_std <= 0:
            cicle_std = cicle * 0.30
        return cicle, cicle_std, 'personal', '#1565C0', '#1565C0'
    else:
        pop = pop_cycle[pop_cycle['Familia_Potencial'] == familia]
        cicle     = float(pop.iloc[0]['pop_cicle_mig']) if len(pop) > 0 else 45.0
        cicle_std = float(pop.iloc[0]['pop_cicle_std']) if len(pop) > 0 else 15.0
        if pd.isna(cicle_std) or cicle_std <= 0:
            cicle_std = cicle * 0.30
        return cicle, cicle_std, 'poblacional', '#E65100', '#E65100'


def _update_slider_for_client(client_id, familia):
    """Ajusta el rang i valor inicial del slider quan canvia el client."""
    cdata = _get_client_timeline(client_id, familia)
    if len(cdata) == 0:
        return
    primer_date   = cdata['primer_pedido'].iloc[0]
    dies_avui_real = (REFERENCE_DATE - primer_date).days
    # Rang: des del primer gap possible fins l'avui real
    first_gap = int(cdata['dies_des_del_primer'].iloc[1]) if len(cdata) > 1 else 10
    dia_avui_w.min   = max(first_gap, 1)
    dia_avui_w.max   = dies_avui_real
    # Valor inicial: ~60% del timeline per tenir compres futures visibles
    initial = max(int(dies_avui_real * 0.60), dia_avui_w.min)
    dia_avui_w.value = initial


def on_familia_change(change):
    new_clients = get_clients_for_familia(change['new'])
    client_w.options = new_clients
    if new_clients:
        client_w.value = new_clients[0]
        _update_slider_for_client(new_clients[0], change['new'])

def on_client_change(change):
    _update_slider_for_client(change['new'], familia_w.value)

familia_w.observe(on_familia_change, names='value')
client_w.observe(on_client_change,   names='value')


# ── Funció principal de visualització ─────────────────────────────────────────
def plot_client_timeline(client_id, familia, min_pedidos_establert, dia_avui_sim):
    cdata = _get_client_timeline(client_id, familia)
    if len(cdata) == 0:
        print(f'⚠️  Client {client_id} sense dades per a {familia}')
        return

    primer_date    = cdata['primer_pedido'].iloc[0]
    dies_avui_real = (REFERENCE_DATE - primer_date).days

    # ── Split: historial conegut vs futur real ────────────────────────────────
    known  = cdata[cdata['dies_des_del_primer'] <= dia_avui_sim]
    future = cdata[cdata['dies_des_del_primer'] >  dia_avui_sim]
    n_known = len(known)

    if n_known == 0:
        print('⚠️  Mou el slider cap a la dreta: no hi ha cap compra coneguda.')
        return

    dies_ultim_known = int(known['dies_des_del_primer'].max())
    es_nou = n_known < min_pedidos_establert

    # ── Cicle calculat SENSE data leakage ────────────────────────────────────
    cicle, cicle_std, cicle_tipus, bar_color, cicle_color = _get_cycle(known, familia, es_nou)

    # ── Zones de predicció des del darrer pedido conegut ──────────────────────
    preds = []
    for offset in range(1, 12):
        nd = dies_ultim_known + offset * cicle
        if nd > dies_ultim_known + 5 * cicle:
            break
        preds.append({'day': nd, 'low': nd - 0.5*cicle_std, 'high': nd + 0.5*cicle_std,
                      'risk_high': nd + 1.5*cicle_std})

    # ── Comprovar encerts (compres reals dins zona verda) ─────────────────────
    hit_set = set()
    for _, row in future.iterrows():
        d = row['dies_des_del_primer']
        if any(p['low'] <= d <= p['high'] for p in preds):
            hit_set.add(d)

    n_future = len(future)
    n_hits   = len(hit_set)
    hit_rate = n_hits / n_future if n_future > 0 else None

    future_hit  = future[future['dies_des_del_primer'].isin(hit_set)]
    future_miss = future[~future['dies_des_del_primer'].isin(hit_set)]

    # ── PLOT ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(15, 5))
    ymax  = float(cdata['euros'].max()) if cdata['euros'].max() > 0 else 1.0
    bar_w = max(cicle * 0.05, 3)

    # 1. Zones de predicció (fons)
    for p in preds:
        ax.axvspan(p['low'],      p['high'],      alpha=0.14, color='green', zorder=1)
        ax.axvspan(p['high'],     p['risk_high'], alpha=0.08, color='red',   zorder=1)
        ax.axvline(p['day'], color=cicle_color, ls='--', lw=0.9, alpha=0.5, zorder=2)

    # 2. Barres historial conegut
    ax.bar(known['dies_des_del_primer'], known['euros'],
           width=bar_w, color=bar_color, alpha=0.85, zorder=3)

    # 3. Compres futures reals: ✅ encert (verd) / ❌ no encert (gris)
    if len(future_hit) > 0:
        ax.bar(future_hit['dies_des_del_primer'], future_hit['euros'],
               width=bar_w, color='#2E7D32', alpha=0.75, zorder=4,
               label=f'✅ Encert (n={len(future_hit)})')
    if len(future_miss) > 0:
        ax.bar(future_miss['dies_des_del_primer'], future_miss['euros'],
               width=bar_w, color='#90A4AE', alpha=0.55, zorder=3,
               label=f'❌ No encert (n={len(future_miss)})')

    # 4. Línia "avui simulat"
    ax.axvline(dia_avui_sim, color='black', lw=2.4, zorder=6)

    # 5. Etiquetes d'import
    for _, row in known.iterrows():
        if row['euros'] > 0:
            ax.text(row['dies_des_del_primer'], row['euros'] + ymax*0.015,
                    f'{row["euros"]:.0f}€', ha='center', va='bottom', fontsize=7.5, color='#222')
    for _, row in future.iterrows():
        if row['euros'] > 0:
            c = '#1B5E20' if row['dies_des_del_primer'] in hit_set else '#546E7A'
            ax.text(row['dies_des_del_primer'], row['euros'] + ymax*0.015,
                    f'{row["euros"]:.0f}€', ha='center', va='bottom', fontsize=7.5, color=c)

    # ── Títol ─────────────────────────────────────────────────────────────────
    estat = '🆕 NOU' if es_nou else '✅ ESTABLERT'
    if hit_rate is not None:
        hit_txt = f'  ·  Taxa encert zona verda: {n_hits}/{n_future} = {hit_rate*100:.0f}%'
    else:
        hit_txt = '  ·  No hi ha compres futures per validar'

    ax.set_title(
        f'Client {client_id}  ·  {familia}  ·  {estat}  ·  '
        f'Cicle {cicle_tipus}: {cicle:.0f} dies (±{cicle_std:.0f}){hit_txt}\n'
        f'🔵 Historial fins dia {dia_avui_sim}  ·  '
        f'🟢 Encerts  ·  🔲 No encerts  ·  ⬛ Avui simulat',
        fontsize=10, pad=8
    )
    ax.set_xlabel('Dies des del primer pedido  (dia 0 = primera compra)', fontsize=10)
    ax.set_ylabel('Import (€)', fontsize=10)

    # Llegenda
    handles = [
        mpatches.Patch(color=bar_color, alpha=0.85,
                       label=f'Historial conegut ({n_known} pedidos, cicle {cicle_tipus})'),
        mpatches.Patch(color='green',   alpha=0.3,  label='Finestra esperada (±0.5σ)'),
        mpatches.Patch(color='red',     alpha=0.2,  label='Zona de risc (>0.5σ retard)'),
        plt.Line2D([0],[0], color='black', lw=2,    label=f'Avui simulat (dia {dia_avui_sim})'),
    ]
    if len(future_hit) > 0:
        handles.append(mpatches.Patch(color='#2E7D32', alpha=0.75,
                                      label=f'✅ Encert ({len(future_hit)})'))
    if len(future_miss) > 0:
        handles.append(mpatches.Patch(color='#90A4AE', alpha=0.55,
                                      label=f'❌ No encert ({len(future_miss)})'))
    ax.legend(handles=handles, loc='upper left', fontsize=8)

    ax.grid(axis='y', alpha=0.3)
    x_right = max(dia_avui_sim * 1.05,
                  dies_ultim_known + 3.5 * cicle,
                  cdata['dies_des_del_primer'].max() * 1.03)
    ax.set_xlim(-bar_w * 2, x_right)
    ax.set_ylim(0, ymax * 1.20)
    plt.tight_layout()
    plt.show()

    # ── Resum text ────────────────────────────────────────────────────────────
    print(f"{'─'*58}")
    print(f"  Backtesting  |  Client {client_id}  |  {familia}")
    print(f"{'─'*58}")
    print(f"  Avui simulat:        dia {dia_avui_sim}  "
          f"(data real: {(primer_date + pd.Timedelta(days=dia_avui_sim)).strftime('%d/%m/%Y')})")
    print(f"  Historial conegut:   {n_known} pedidos  (fins dia {dies_ultim_known})")
    print(f"  Cicle calculat:      {cicle:.0f} dies ±{cicle_std:.0f}  ({cicle_tipus})")
    print(f"  Compres futures:     {n_future}")
    if hit_rate is not None:
        stars = '⭐' * min(int(hit_rate * 5), 5)
        print(f"  ✅ Encerts:          {n_hits} / {n_future}  ({hit_rate*100:.0f}%)  {stars}")
        print(f"  ❌ No encerts:       {n_future - n_hits} / {n_future}")
    else:
        print(f"  (Mou el slider cap a l'esquerra per generar compres futures a validar)")
    print(f"{'─'*58}")


# ── Callbacks ──────────────────────────────────────────────────────────────────
def update_chart(change=None):
    with out:
        clear_output(wait=True)
        if client_w.value is not None:
            plot_client_timeline(
                client_w.value, familia_w.value,
                min_pedidos_w.value, dia_avui_w.value
            )

familia_w.observe(update_chart,     names='value')
client_w.observe(update_chart,      names='value')
min_pedidos_w.observe(update_chart, names='value')
dia_avui_w.observe(update_chart,    names='value')


# ── Layout ─────────────────────────────────────────────────────────────────────
header = widgets.HTML(
    '<h3 style="margin:4px 0;color:#1565C0">🔍 Explorador + Backtesting interactiu</h3>'
    '<p style="color:#555;margin:2px 0">'
    'Mou <b>"Avui simulat"</b> cap a l\'esquerra per anar enrere en el temps. '
    'El model prediu amb l\'historial conegut fins aquell dia i et mostra si les compres reals '
    'cauen a les zones verdes (✅ encert) o fora (❌). '
    'El cicle es recalcula <i>sense data leakage</i>.</p>'
)
row1 = widgets.HBox([familia_w, client_w, min_pedidos_w],
                    layout=widgets.Layout(gap='10px', align_items='center'))
row2 = widgets.HBox([dia_avui_w],
                    layout=widgets.Layout(margin='4px 0 0 0'))
display(widgets.VBox([header, row1, row2, out]))

# Inicialitzar slider per al primer client i renderitzar
_update_slider_for_client(client_w.value, familia_w.value)
update_chart()
